In [1]:
import numpy as np

## Conversions

In [2]:
AU_to_km = 1.495978707e8 # AU to Km
day_to_sec = 86400.0 # Day to Seconds
arcsec_to_rad = np.deg2rad(1/3600) # Arcsecond to Radian

## Constants

In [3]:
sun_radius = 695700 # Radius of the Sun (km)
sun_mu = 1.32712440018e11  # Gravitational parameter of the Sun (km^3/s^2)

R_earth = 6378.137 # Radius of the Earth(km)
R_earth_polar = 6356.7523 # Polar radius of the Earth(km)

c_km_s = 299792.458 #Speed of light (s)

## Code

![](https://cdn.hackclub.com/019d41e8-3c3d-7b1b-b982-1c8db02d2431/Screenshot%202026-03-22%20173922.png)

``` mermaid

    flowchart LR

        %% 1. Style Definitions
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions
        subgraph N0["Minor Planet Center Requests"]
            N0a[get-orb]
        end

        subgraph N1["Minor Planet Center Responses"]
            N1a[orb]:::var
            D1a["Eros's Orbital Data"]:::ghost
        end

        subgraph N2[" "]
            N2a[x]:::var
            N2b[coefficient values like]:::ghost
            N2c[x, y, z, vx, vy, vz]:::quantity
        end

        %% 3. Node Definitions
        N3a[t0_mjd]:::var

        %% 4. Box Hiding 
        N2:::ghost
    
        N0a --mpc_orb--> N1a

        N1a --coefficient_values--> N2a
        N1a --epoch--> N3a

        N2a --> N2b --> N2c
        N2c --Convert Au/day to km/s and Au to km--> N2a
```

In [4]:
import requests

response = requests.get("https://data.minorplanetcenter.net/api/get-orb", json={"desig": "Eros"})

if response.ok:
    orb = response.json()[0]['mpc_orb']
else:
    print("Error: ", response.status_code, response.content)

In [5]:
orbital_elements = orb[0]['CAR']['coefficient_values']
#coefficient values x, y, z, vx, vy, vz - state vector
x = np.array(orbital_elements) 

In [6]:
x[:3] *= AU_to_km #convert to km
x[3:] *= AU_to_km / day_to_sec #convert to km/s

In [7]:
print(x)

[-1.73383024e+08 -1.03230896e+08 -3.85253329e+07  8.35317115e+00
 -2.47405348e+01 -1.34416511e+00]


In [8]:
t0_mjd = orb[0]['epoch_data']['epoch'] #mjd timestamp of state vector

![](https://cdn.hackclub.com/019d41e8-917b-7bcf-b4d8-77ade07498a3/Screenshot%202026-03-22%20184824.png)

``` mermaid

    flowchart LR

        %% 1. Style Definitions
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef userfunc fill:#2e1a3e,color:#d8a8d8stroke:#8a4a8a,stroke-width:2px
        classDef builtinfunc fill:#3e1a1a,color:#d8a8a8,stroke:#8a4a4a,stroke-width:2px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions
        subgraph N0["Minor Planet Center Requests"]
            N0b[get-obs]
        end

        subgraph N1["Minor Planet Center Responses"]
            N1b["Eros's Orbital Data"]:::ghost
        end

        subgraph N6["NAIF Kernels"]
            subgraph N6a["lsk/"]
                N6aa[naif0012.tls]
            end
            subgraph N6b["spk/"]
                N6ba[planets/de440.bsp]
                N6bb['asteroids/codes_300ast_20100725.tf']
            end
            subgraph N6c["pck/"]
                N6ca[earth_latest_high_prec.bpc]
                N6cb[gm_de431.tpc]
                N6cc[pck00010.tpc]
            end
        end

        subgraph N14[n_body_ode]
            direction LR
            subgraph N14a["Required"]
                N14aa[t]:::var
                N14ab[state]:::var
            end

            N14b[r]:::var
            N14c[v]:::var

            N14d["acceleration calculation"]:::ghost
            N14e[a]:::var

            subgraph N14f[" "]
                N14fa[ssbs]:::var
                N14fb["gravitational parameter values of planets, major asteroids, sun and moon as tuple"]:::ghost
                N14fc[name, mu]:::quantity
            end

            N14g["calculate gravitational pull due to sun"]:::ghost 

            N14h["subtract gravitational pull from a"]:::ghost 

            subgraph N14i["Returns"]
                N14ia[coefficient values like]:::ghost
                N14ib[vx, vy, vz, ax, ay, az]:::quantity
            end
        end

        subgraph N16[get_stn_properties]
            direction LR
            subgraph N16a["Requirements"]
                N16aa[stn]:::var
            end

            N16b["Reads stations CSV"]:::ghost

            subgraph N16c["Returns"]
                N16ca[dictionary with]:::ghost
                N16cb[Long., cos, sin]:::quantity
            end
        end

        subgraph N17["get_observer_pos_j2000"]
            direction LR
            subgraph N17a["Requirements"]
                N17aa[stn]:::var
                N17ab[t_obs]:::var
                N17ac[properties]:::var
                N17ad[stn_to_ecef]:::userfunc
            end

            N17b["Rotate to J2000"]:::ghost

            subgraph N17c["Returns"]
                N17ca[coefficient values like]:::ghost
                N17cb[x, y, z]:::quantity
            end
        end

        subgraph N18["Table values from relevant papers"]
            N18a["Astrometric Uncertainity"]
        end

        subgraph N23["loadVFCC"]
            direction LR
            subgraph N23a["Requirements"]
                N23aa[details]:::var
                N23ab[dec_obs]:::var
                N23ac[lookup]:::var
            end

            N23b["Get lowest sigma values"]:::ghost

            subgraph N23c["Returns"]
                N23ca[sigma_ra]:::var
                N23cb[sigma_dec]:::var
            end
        end

        subgraph N24["astrometric_error"]
            direction LR
            subgraph N24a["Requirements"]
                N24aa[details]:::var
                N24ab[dec_obs]:::var
                N24ac[lookup]:::var
            end

            N24b["Get Root Mean Square values in radians"]:::ghost

            subgraph N24c["Returns"]
                N24ca[sigma_ra_rad]:::var
                N24cb[sigma_dec_rad]:::var
            end
        end

        subgraph N27["ObservationalError"]
            direction LR
            subgraph N27a["Requirements"]
                N27aa[obs]:::var
                N27ab[trajectory solution]:::var
            end

            subgraph N27b[" "]
                N27ba[t_obs]:::var
                subgraph N27bb[" "]
                    N27bba[ra_obs]:::var
                    N27bbb[dec_obs]:::var
                end
            end

            subgraph N27c[" "]
                N27ca[state_at_obs]:::var
                N27cb[r_asteroid]:::var
            end

            N27d[stn]:::var
            N27e[astcat]:::var

            N27f[stn_properties]:::var
            N27g[obs_pos]:::var

            N27h[ra_pred]:::var
            N27i[dec_pred]:::var

            N27j[rmsra]:::var
            N27k[rmsdec]:::var

            N27l[telescope_details]:::var

            subgraph N27m["Returns"]
                N27ma[residual]:::var
                N27mb[weight]:::var
            end

        end


        %% 3. Node Definitions
        N2[x]:::var
        N5[t0_mjd]:::var
        N7[sp.furnsh]:::builtinfunc
        N8[obsdata]:::var
        N9[t_start]:::var
        N10[t_end]:::var
        N11[t_epoch]:::var
        N12[t0]:::var
        N13[ecl_to_j2000]:::var
        N15[trajectory solution]:::var
        N19[VFCCLookupDefault]:::var
        N20[VFCCAstcat]:::var
        N21[VFCCStn]:::var
        N22[VFCCLookup]:::var
        N25[todays date]:::ghost
        N26[jd_today]:::var
        N28[results]:::var
        N29[residual_states]:::var
        N30[weights]:::var

        N1000["2.1"]

        %% 4. Style Definitions
        N14:::userfunc
        N14f:::ghost
        N16:::userfunc
        N17:::userfunc
        N23:::userfunc
        N24:::userfunc
        N27:::userfunc
        N27b:::ghost
        N27bb:::ghost
        N27c:::ghost
        
        %% 5. Graphing
        N0b --Convert obstime to ET--> N8

        N1000 --coefficient_values--> N2
        N1000 --epoch--> N5

        N2 --Rotate to J2000--> N13
        N2 --Propagate to start--> N2 
        N2 --DOP853 + 8 Step Runge-Kutta solver--> N15

        N5 --Convert JDTDT to ET-->N11

        N6 --check_download--> N6
        N6 --planetaryMetaK.txt--> N7

        N8 --> N0b
        N8 -- get nearest recorded start date --> N9
        N8 -- get last date --> N10
        N8 --> N2

        N9 --> N8

        N10 --> N8

        N12 --Rotate to J2000--> N13

        N13 --> N2

        N14aa --> N14g
        N14ab --radii--> N14b
        N14ab --velocities--> N14c
        N14b --> N14d
        N14c --> N14ia
        N14d --> N14e
        N14e --> N14h
        N14e --> N14ia
        N14fa --> N14fb
        N14fa --> N14g
        N14fb --> N14fc
        N14fc --> N14fa
        N14g --> N14e
        N14h --> N14e
        N14ia --> N14ib
        N14 --> N15

        N15 --> N14
        N15 --> N27ab

        N16a --> N16b
        N16b --> N16ca
        N16ca --> N16cb
        N16cb --> N27f

        N17a --> N17b
        N17b --> N17ca
        N17ca --> N17cb
        N17cb --> N27g

        N18a --> N19
        N18a --> N20
        N18a --> N21

        N19 --> N22

        N20 --> N22

        N21 --> N22

        N23 --> N24ac
        N23aa --> N23b
        N23ab --> N23ca
        N23ac --> N23b
        N23b --> N23ab
        N23b --> N23cb

        N24a --> N24b
        N24b --> N24c
        N24ca --> N27mb
        N24cb --> N27mb

        N25 --> N26

        N26 --past 90 days--> N9
        N26 --> N10

        N27 --Loop through every observation-->N27
        N27 --> N28
        N27aa --> N27ba
        N27aa --> N27ca
        N27aa --stn--> N27d
        N27aa --astcat--> N27e
        N27aa --rmsra--> N27j
        N27aa --rmsdec--> N27k
        N27ab --> N27ca
        N27ba --> N27bba
        N27ba --> N17ab
        N27ba --> N27bbb
        N27bba --> N27ma
        N27bbb --> N24ab
        N27bbb --> N27ma
        N27ca --positions--> N27cb
        N27cb --Light Time correction--> N27cb
        N27cb --> N27h
        N27cb --> N27i
        N27d --> N16aa
        N27d --> N17aa
        N27d --> N27l
        N27e --> N27l
        N27f --> N17ac
        N27f --Use spkezr if no station properties, and extract positions--> N27g
        N27g --> N27h
        N27g --> N27i
        N27h --> N27ma
        N27i --> N27ma
        N27j --> N27l
        N27k --> N27l
        N27l --> N24aa

        N28 --> N29
        N28 --> N30
```

$
a_{small} = \frac{G_{large}}{r^2} = \frac{\mu}{r^2}
$

### Newton's Universal Law of Gravitation in 3D
$
\vec{a_{small}} = \frac{Gm_{large}}{r^2} = \frac{\mu}{r^2}
$

In [9]:
kernels = {
    "lsk/": ['naif0012.tls',],
    
    "spk/": ['planets/de440.bsp', 
            'asteroids/codes_300ast_20100725.tf',
            'asteroids/codes_300ast_20100725.bsp',],
    
    "pck/": ['earth_latest_high_prec.bpc', 
            'gm_de431.tpc',
            'pck00010.tpc',]
}

In [10]:
import os
import urllib.request

url = 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/'
kernels_loaded = os.listdir("../data/kernels")

for kernel_type, items in kernels.items():
    for item in items:
        item_name = item.split('/')[-1]
        if item_name in kernels_loaded:
            print(f"{item_name} aldready downloaded")
            continue
        else:
            print
            download_url = url + kernel_type + item
            print(f"Downloading {item_name} from {download_url}")
            urllib.request.urlretrieve(download_url, f"kernels/{item_name}")
            
print("All files done!")
        

naif0012.tls aldready downloaded
de440.bsp aldready downloaded
codes_300ast_20100725.tf aldready downloaded
codes_300ast_20100725.bsp aldready downloaded
earth_latest_high_prec.bpc aldready downloaded
gm_de431.tpc aldready downloaded
pck00010.tpc aldready downloaded
All files done!


In [11]:

import spiceypy as sp

# The meta kernel file contains entries pointing to the following SPICE kernels, which the user needs to download.
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440.bsp
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/earth_latest_high_prec.bpc
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/asteroids/codes_300ast_20100725.tf
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc
#   https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/asteroids/codes_300ast_20100725.bpc

#   The following is the contents of a metakernel that was saved with
#   the name 'planetaryMetaK.txt'.
#   \begindata
#   KERNELS_TO_LOAD=(
#     'kernels/naif0012.tls',
#     'kernels/de440.bsp',
#     'kernels/earth_latest_high_prec.bpc',
#     'kernels/gm_de431.tpc',
#     'kernels/codes_300ast_20100725.tf',
#     'kernels/pck00010.tpc',
#     'kernels/codes_300ast_20100725.bpc'
#   )
#   \begintext

sp.kclear()
sp.furnsh("../data/planetaryMetaK.txt")

$
F_g = -Gm_i\sum_{j=1, j \neq i}^n\frac{m_j}{r_{ji}^3}(r_{ji})
$

$
a_g = -\mu\sum_{}\frac{r_{ji}}{|r_{ji}|^3}
$

In [12]:
def n_body_ode(t, state):
    
    eros_position = state[:3]
    eros_velocity = state[3:]
    
    r = eros_position
    v = eros_velocity
    
    r_norm = np.linalg.norm(r)
    a = -sun_mu * r / r_norm**3
    
    ssbs = [
        # (name, mu)
        
        # Terrestrial
        ('1', 22031.78000000002), # MERCURY
        ('2',   324858.592), # VENUS
        ('3',   398600.435), # EARTH
        ('4',    42828.375), # MARS
        
        # Satellites
        ('301',               4902.800), # MOON
        
        # Asteroids
        ('2000001',              63.129999999999995),
        ('2000002',              13.73),
        ('2000004',              17.28999999999999),
        
        ('2000010',            5.78),               # Hygiea
        ('2000015',            2.10),               # Eunomia
        ('2000016',            1.81),               # Psyche
        ('2000029',            0.86),               # Amphitrite
        ('2000052',            1.5899999999999999), # Europa
        ('2000065',            0.9099999999999999), # Cybele
        ('2000087',            0.9899999999999999), # Sylvia
        ('2000088',            1.02),               # Thisbe
        ('2000511',            2.259999999999999),  # Davida
        ('2000704',            2.189999999999999),  # Interamnia
        
        # Jovian
        ('5', 126712764.1), # JUPITER
        ('6',  37940585.2), # SATURN
        ('7',  5794548.600000008), # URANUS
        ('8', 6836527.100580023), # NEPTUNE
    ]
    
    for name, mu in ssbs:
        
        # i is planet, j is eros
        
        planet, _ = sp.spkezr(name, t, 'J2000', 'NONE', 'SUN') #calculates state vector of planet
        
        r_i = planet[:3]
        r_i_norm = np.linalg.norm(r_i)
        
        r_ji = r - r_i
        r_ji_norm = np.linalg.norm(r_ji)
        
        # subtracting pull due to sun
        a += -mu * (r_ji / r_ji_norm**3 + r_i / r_i_norm**3)
    
    return np.concatenate((v, a)) # vx, vy, vz, ax, ay, az

In [13]:
from scipy.integrate import solve_ivp

In [14]:
response = requests.get("https://data.minorplanetcenter.net/api/get-obs", json={"desigs": ["Eros"], "output_format": ["ADES_DF"]})

if response.ok:
    obs_data = response.json()[0]['ADES_DF']
else:
    print("Error: ", response.status_code, response.content)


In [15]:
for n, obs in enumerate(obs_data): #convert to ephemeris time
    obs['obstime'] = sp.str2et(obs['obstime'])
    obs_data[n] = obs

In [16]:
from datetime import datetime, timedelta

jd_today = datetime.today().date() #todays date in jd

t_epoch = sp.unitim(t0_mjd + 2400000.5, 'JDTDT', 'ET')  # convert mjd to jd
t_start = sp.str2et(str(jd_today - timedelta(days=90))) # Last 90 days
t_end = jd_today

for n, obs in enumerate(obs_data):
    if obs['obstime'] >= t_start:
        t_start = obs_data[n]['obstime'] #propagating from nearest t_start datapoint
        t_end = obs_data[-1]['obstime'] #naturally t_end would be the last datapoint
        break
    
total = len(obs_data)
start_index = n
    
print(f"Propagating from {start_index}th observation at {t_start} to {total}th index at {t_end} as a total of {total - n} observations")

Propagating from 18022th observation at 828002436.3956475 to 18553th index at 835359267.7043719 as a total of 531 observations


In [17]:
ecl_to_j2000 = sp.pxform('ECLIPJ2000', 'J2000', t_epoch)  #calculate state at t_start by propagating it from t_epoch to t_start
x = np.concatenate([ecl_to_j2000 @ x[:3], ecl_to_j2000 @ x[3:]])
x = solve_ivp(n_body_ode,
                (t_epoch, t_start),
                x,
                method = "DOP853",
                rtol=1e-12,
                atol=1e-12,
                dense_output=True,).y[:, -1]

In [18]:
trajectory_solution = solve_ivp(n_body_ode, #precompute entire trajectory from t_start to t_end
                                (t_start, t_end),
                                x,
                                method = "DOP853",
                                rtol=1e-12,
                                atol=1e-12,
                                dense_output=True,).sol

In [19]:
# stations.csv downloaded from https://www.minorplanetcenter.net/iau/lists/ObsCodes.html

import csv

def get_stn_properties(stn):
    
    with open('../data/stations.csv', 'r', newline='\n') as stns:
        reader = csv.DictReader(stns)
        for row in reader:
            if row['Code'] == stn:
                return {
                    'Long.': float(row['Long.']) if row['Long.'].strip() != '' else None,
                    'cos': float(row['cos']) if row['cos'].strip() != '' else None,
                    'sin': float(row['sin']) if row['sin'].strip() != '' else None,
                }
    return None

In [20]:
def stn_to_ecef(stn, properties):
    
    lon = np.deg2rad(properties['Long.'])
    x = R_earth * properties['cos'] * np.cos(lon)
    y = R_earth * properties['cos'] * np.sin(lon)
    z = R_earth * properties['sin']
    
    return np.array([x, y, z])

In [21]:
def get_observer_pos_j2000(stn, t_obs, properties):
    r_ecef = stn_to_ecef(stn, properties)
        
    earth, _ = sp.spkezr('EARTH', t_obs, 'J2000', 'NONE', 'SUN')
    rot = sp.pxform('ITRF93', 'J2000', t_obs)
    r_obs_j2000 = rot @ r_ecef
    
    return earth[:3] + r_obs_j2000

In [22]:
VFCCLookupDefault = {
# (stn, astcat) : (ra, dec) arcsec 

# USNOA2.0 
('704', 'USNOA2')  : (0.63, 0.60),
('699', 'USNOA2')  : (0.62, 0.53),
('691', 'USNOA2')  : (0.30, 0.30),
('608', 'USNOA2')  : (0.61, 0.75),
('703', 'USNOA2')  : (0.69, 0.63),
('644', 'USNOA2')  : (0.29, 0.30),
('291', 'USNOA2')  : (0.46, 0.32),
('599', 'USNOA2')  : (0.39, 0.34),
('333', 'USNOA2')  : (0.55, 0.53),
('D35', 'USNOA2')  : (0.39, 0.38),

# USNOA1.0 
('704', 'USNOA1')  : (0.76, 0.73),
('691', 'USNOA1')  : (0.49, 0.46),

# USNOB1.0 
('699', 'USNOB1')  : (0.61, 0.54),
('644', 'USNOB1')  : (0.24, 0.20),
('691', 'USNOB1')  : (0.30, 0.28),
('291', 'USNOB1')  : (0.39, 0.26),

# UCAC1
('703', 'UCAC1')   : (0.63, 0.59),
('G96', 'UCAC1')   : (0.32, 0.27),
('E12', 'UCAC1')   : (0.50, 0.45),
('683', 'UCAC1')   : (0.79, 0.90),
('J75', 'UCAC1')   : (0.41, 0.37),
('106', 'UCAC1')   : (0.40, 0.39),
('143', 'UCAC1')   : (0.57, 0.47),

# UCAC2
('703', 'UCAC2')   : (0.63, 0.59),
('G96', 'UCAC2')   : (0.32, 0.27),
('E12', 'UCAC2')   : (0.50, 0.45),
('683', 'UCAC2')   : (0.79, 0.90),
('J75', 'UCAC2')   : (0.41, 0.37),
('106', 'UCAC2')   : (0.40, 0.39),
('143', 'UCAC2')   : (0.57, 0.47),

# Gaia2
('T14', 'Gaia2')   : (0.10, 0.10),
('T12', 'Gaia2')   : (0.10, 0.10),
('T09', 'Gaia2')   : (0.10, 0.10),
('Y28', 'Gaia2')   : (0.30, 0.30),
('568', 'Gaia2')   : (0.10, 0.10),
('G83', 'Gaia2')   : (0.20, 0.20),
('309', 'Gaia2')   : (0.20, 0.20),

# Gaia3
('T14', 'Gaia3')   : (0.10, 0.10),
('T12', 'Gaia3')   : (0.10, 0.10),
('T09', 'Gaia3')   : (0.10, 0.10),
('Y28', 'Gaia3')   : (0.30, 0.30),
('568', 'Gaia3')   : (0.10, 0.10),
('G83', 'Gaia3')   : (0.20, 0.20),
('309', 'Gaia3')   : (0.20, 0.20),

# Gaia3E
('T14', 'Gaia3E')  : (0.10, 0.10),
('T12', 'Gaia3E')  : (0.10, 0.10),
('T09', 'Gaia3E')  : (0.10, 0.10),
('Y28', 'Gaia3E')  : (0.30, 0.30),
('568', 'Gaia3E')  : (0.10, 0.10),
('G83', 'Gaia3E')  : (0.20, 0.20),
('309', 'Gaia3E')  : (0.20, 0.20),

# Tycho-2
('689', 'Tycho-2') : (0.20, 0.21),

}

VFCCAstcat = {
# (astcat) : (ra, dec) arcsec 

'Tycho-2'          : (0.24, 0.25),
'UCAC2'            : (0.53, 0.49),
'UCAC1'            : (0.53, 0.49),
'UCAC4'            : (0.30, 0.30),
'USNOB1'           : (0.48, 0.42),
'USNOA1'           : (0.72, 0.69),
'USNOA2'           : (0.61, 0.58),
'Gaia2'            : (0.20, 0.20),
'Gaia3'            : (0.20, 0.20),
'Gaia3E'           : (0.20, 0.20),
'ATLAS2'           : (0.20, 0.20),

}

VFCCStn = {
# (stn) : (ra, dec) arcsec 

'645'              : (0.30, 0.30),
'673'              : (0.30, 0.30),
'689'              : (0.50, 0.50),
'950'              : (0.50, 0.50),
'H01'              : (0.30, 0.30),
'J04'              : (0.40, 0.40),
'W84'              : (0.50, 0.50),
'LCO'              : (0.40, 0.40),

}

In [23]:
VFCCLookup = {
    
    "Default" : VFCCLookupDefault,
    "Astcat"  : VFCCAstcat,
    "Station" : VFCCStn,
    
}

![Uncertainties - Veres et al.](https://cdn.hackclub.com/019d41f5-4f49-79fa-907e-4670c4fbb91b/Screenshot%202026-03-27%20173647.png)

![Uncertainties - Veres et al.](https://cdn.hackclub.com/019d41f5-e298-71e5-8a0f-a76e04ae62f4/Screenshot%202026-03-27%20215631.png)

![Uncertainties - ATLAS2 Paper](https://cdn.hackclub.com/019d41f6-4bb3-7d34-9d56-997bc03d7b66/Screenshot%202026-03-27%20222123.png)

In [24]:
def loadVFCC(details, dec_obs, lookup=VFCCLookup):
    
    if (details["stn"], details["astcat"]) in VFCCLookup["Default"]:
        
        sigma_ra  = VFCCLookup["Default"][(details["stn"], details["astcat"])][0]
        sigma_dec = VFCCLookup["Default"][(details["stn"], details["astcat"])][1]
        
    elif details["stn"] in VFCCLookup["Station"] and details["astcat"] in VFCCLookup["Astcat"]:
        
        sigma_ra  = VFCCLookup["Astcat"][details["astcat"]][0]
        sigma_dec = VFCCLookup["Astcat"][details["astcat"]][1]
        
        if sigma_ra < VFCCLookup["Station"][details["stn"]][0]:
            sigma_ra = VFCCLookup["Station"][details["stn"]][0]
            
        if sigma_dec < VFCCLookup["Station"][details["stn"]][1]:
            sigma_dec = VFCCLookup["Station"][details["stn"]][1]
        
    elif details["astcat"] in VFCCLookup["Astcat"]:
        
        sigma_ra  = VFCCLookup["Astcat"][details["astcat"]][0]
        sigma_dec = VFCCLookup["Astcat"][details["astcat"]][1]
        
    elif details["stn"] in VFCCLookup["Station"]:
        
        sigma_ra  = VFCCLookup["Station"][details["stn"]][0]
        sigma_dec = VFCCLookup["Station"][details["stn"]][1]
        
    else:
        
        sigma = 0.2
        
        sigma_ra  = sigma * np.cos(dec_obs)
        sigma_dec = sigma
        
    return {"sigma_ra": sigma_ra, "sigma_dec": sigma_dec}

In [25]:
def astrometric_error(details, dec_obs, arcsec_to_rad):
    
    sigma_ra  = None
    sigma_dec = None
    
    def rms(sigma_ra, sigma_dec):
        
        return {
            "rmsra"  : 1 / sigma_ra ** 2,
            "rmsdec" : 1 / sigma_dec ** 2}
            
    def largeMPCObserverError(details):
        
        sigma_ra  = float(details["rmsra"])
        sigma_dec = float(details["rmsdec"])
        
        if sigma_ra <= 0.2 and sigma_dec <= 0.2:
            return False
        else:
            return True
            
    if details["rmsra"] is not None and details["rmsdec"] is not None:
        
        sigma_ra  = float(details["rmsra"])
        sigma_dec = float(details["rmsdec"])
            
        if largeMPCObserverError(details):
            
            VFCCUncertainties = loadVFCC(details, dec_obs)
            
            sigma_ra  = VFCCUncertainties["sigma_ra"]
            sigma_dec = VFCCUncertainties["sigma_dec"]
            
    else:
        
        VFCCUncertainties = loadVFCC(details, dec_obs)
            
        sigma_ra  = VFCCUncertainties["sigma_ra"]
        sigma_dec = VFCCUncertainties["sigma_dec"]
        
    sigma_ra_rad  = sigma_ra * arcsec_to_rad
    sigma_dec_rad = sigma_dec * arcsec_to_rad
        
    rms_values = rms(sigma_ra=sigma_ra_rad, sigma_dec=sigma_dec_rad)
    
    return [rms_values["rmsra"], rms_values["rmsdec"]]

In [26]:
def angle_diff(a, b):
    return (a - b + np.pi) % (2*np.pi) - np.pi

In [27]:
def ObservationalError(obs, trajectory_solution=trajectory_solution):

    t_obs   = obs['obstime']
    ra_obs  = np.deg2rad(float(obs['ra']))
    dec_obs = np.deg2rad(float(obs['dec']))
    
    state_at_obs = trajectory_solution(t_obs)
    r_asteroid = state_at_obs[:3]
    
    stn    = obs['stn']
    astcat = obs['astcat']
    
    stn_properties = get_stn_properties(stn)
    obs_pos = 0.0
    
    if stn_properties['sin'] == None and stn_properties['cos'] == None:
        earth, _ = sp.spkezr('EARTH', t_obs, 'J2000', 'NONE', 'SUN')
        obs_pos = earth[:3]
    else:
        obs_pos = get_observer_pos_j2000(stn, t_obs, stn_properties)
        
    lt = 0.0
        
    for _ in range(3):
        
        rho = r_asteroid - obs_pos
        lt = np.linalg.norm(rho)
        t_emit = t_obs - (lt / c_km_s)
        r_asteroid = trajectory_solution(t_emit)[:3]
        
    rho = r_asteroid - obs_pos
    rho_hat = rho / np.linalg.norm(rho)
    
    ra_pred = np.arctan2(rho_hat[1], rho_hat[0])
    ra_pred = np.mod(ra_pred, 2*np.pi)
    dec_pred = np.arcsin(rho_hat[2])
    
    residual = [angle_diff(ra_obs, ra_pred), dec_obs - dec_pred]
    
    rmsra = obs['rmsra']
    rmsdec = obs['rmsdec']
    
    telescope_details = {"stn": stn, "astcat": astcat, "rmsra": rmsra, "rmsdec": rmsdec}
    
    weight = astrometric_error(telescope_details, dec_obs, arcsec_to_rad)

    return residual, weight

In [28]:
observations = [obs for obs in obs_data[start_index+2:] if obs['stn'] != '270'] # omits and garbage observations from stn 270 and obs 1, 2
result = [ObservationalError(obs) for obs in observations]

In [29]:
residual_states, weights = zip(*result)
residual_states = np.array(residual_states, dtype=np.float64)
weights         = np.array(weights, dtype=np.float64)

![](https://cdn.hackclub.com/019d41f7-60ad-7f51-8a29-110340875089/Screenshot%202026-03-24%20160634.png)

``` mermaid

    flowchart LR

        %% 1. Style Definitions
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions

        %% 3. Node Definitions
        N1[2.2]
        N2[residual_states]:::var
        N3[weights]:::var
        N4[e_baseline]:::var
        N5[e_t]:::var
        N6[W]:::var
        N7[q]:::var

        %% 4. Box Hiding 

        %% 5. Graphing
        N1 --> N2
        N1 --> N3

        N2 --> N4

        N3 --> N6

        N4 --> N5
        N4 --> N7

        N5 --> N7

        N6 --> N7

```

In [30]:
e_baseline = residual_states.flatten().reshape(-1, 1) # 2*sum(obs_index:), 1
e_t = e_baseline.T # 1, 2*sum(obs_index:)

W = np.diag(weights.flatten()) # 2*sum(obs_index:), 2*sum(obs_index:)
q = e_t @ W @ e_baseline #1, 1

![](https://cdn.hackclub.com/019ed6ce-f96f-7b03-92e4-2715943692b1/image_2026-06-17_234906834.png)

``` mermaid

    flowchart LR

        %% 1. Style Definitions
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef userfunc fill:#2e1a3e,color:#d8a8d8stroke:#8a4a8a,stroke-width:2px
        classDef builtinfunc fill:#3e1a1a,color:#d8a8a8,stroke:#8a4a4a,stroke-width:2px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions
        subgraph N1["get_design_matrix"]
            direction LR
            subgraph N1a["Requires"]
                N1aa[x_current]:::var
                N1ab[e_baseline]:::var
                N1ac[observations_list]:::var
            end

            N1b[num_residuals]:::var

            N1c[perturbations]:::var
            N1d[x_perturbed]:::var

            N1e[perturbed_residuals_list]:::var
            N1f[e_perturbed]:::var

            subgraph N1g["Returns"]
                N1ga[B]:::var
            end
        end

        %% 3. Node Definitions
        N2[2.2]
        N3[n_body_ode]:::userfunc
        N4[t_start]:::var
        N5[t_end]:::var

        %% 4. Style Definitions
        N1:::userfunc 

        %% 5. Graphing
        N1aa --> N1d
        N1ab --> N1b
        N1ab --> N1ga
        N1ac --> N3
        N1ac --> N1e
        N1b --> N1ga
        N1c --> N1d
        N1d --> N3
        N1e --> N1f
        N1f --> N1ga

        N2 --> N3
        N2 --> N4
        N2 --> N5

        N3 --Loop for every observation--> N1ac

        N4 --> N3

        N5 --> N3
```

In [31]:
def get_design_matrix(x_current, e_baseline, observation_list):
    num_residuals = len(e_baseline)
    params = 6
    B = np.zeros((num_residuals, params))

    perturbations = [1.0, 1.0, 1.0, 1e-4, 1e-4, 1e-4]

    for j in range(params):
        x_perturbed = x_current.copy()
        x_perturbed[j] += perturbations[j]
        
        perturbed_trajectory = solve_ivp(
            n_body_ode,
            (t_start, t_end),
            x_perturbed,
            method="DOP853",
            rtol=1e-12,
            atol=1e-12,
            dense_output=True
        ).sol
        
        perturbed_residuals_list = []
        for obs in observations:
            res, _ = ObservationalError(obs, trajectory_solution=perturbed_trajectory)
            perturbed_residuals_list.extend(res)
            
        e_perturbed = np.array(perturbed_residuals_list, dtype=np.float64).reshape(-1, 1)
        
        B[:, j] = ((e_perturbed - e_baseline) / perturbations[j]).flatten()
    
    return B
    

![](https://cdn.hackclub.com/019ed6d0-fc22-7636-a5b6-3ba5001e8b18/image_2026-06-17_235118460.png)

``` mermaid

    flowchart LR

        %% 1. Styles
        classDef var fill:#1a3e1a,color:#a8d8a8,stroke:#4a8a4a,stroke-width:2px
        classDef quantity fill:#2d3748,color:#e2e8f0,stroke:#4a5568,stroke-width:1px
        classDef userfunc fill:#2e1a3e,color:#d8a8d8stroke:#8a4a8a,stroke-width:2px
        classDef builtinfunc fill:#3e1a1a,color:#d8a8a8,stroke:#8a4a4a,stroke-width:2px
        classDef ghost fill:none,stroke:none,color:none

        %% 2. Subgraph Definitions

        %% 3. Node Definitions
        N1[tolerance_dx]:::var
        N2[tolerance_dq]:::var
        N3[Q_old]:::var
        N4[2.2]
        N5[n_body_ode]:::userfunc
        N6[t_start]:::var
        N7[t_end]:::var
        N8[x]:::var
        N9[observations]:::var
        N10[result]:::var
        N11[residual_states]:::var
        N12[e_baseline]:::var
        N13[2.3]
        N14[W]:::var
        N15[Q]:::var
        N16[B]:::var
        N17[C]:::var
        N18[rhs]:::var
        N19[dx]:::var
        N20[sigma]:::var

        %% 4. Style Definitions

        %% 5. Graphing
        N1 --> N19

        N2 --> N19

        N3 --> N15

        N4 --> N5
        N4 --> N6
        N4 --> N7
        N4 --> N8
        N4 --> N9

        N5 --Loop for all observations--> N5
        N5 --> N10

        N6 --> N5

        N7 --> N5

        N8 --> N5
        N8 --> N16

        N9 --> N5
        N9 --> N16

        N10 --> N11

        N11 --> N12

        N12 --> N15
        N12 --> N16
        N12 --> N18

        N13 --> N5
        N13 --> N14
        N13 --> N16

        N14 --> N15
        N14 --> N17
        N14 --> N18

        N15 --Loop over N iterations--> N3

        N16 --> N17
        N16 --> N18

        N17 --Damp--> N17
        N17 --> N19
        N17 --> N20

        N18 --> N19

        N19 --add--> N8
        N19 --check tolerance--> N1
        N19 --check tolerance--> N2
```

In [32]:
max_iterations = 8
tolerance_dx = 1e-3   
tolerance_dQ = 1e-2   

Q_old = float('inf')

for iteration in range(max_iterations):
    
    trajectory_solution = solve_ivp(
        n_body_ode, 
        (t_start, t_end), 
        x, 
        method="DOP853", 
        rtol=1e-12, 
        atol=1e-12, 
        dense_output=True
    ).sol
    
    result = [ObservationalError(obs, trajectory_solution=trajectory_solution) for obs in observations]
    residual_states, _ = zip(*result)
    residual_states = np.array(residual_states, dtype=np.float64)
    
    e_baseline = residual_states.flatten().reshape(-1, 1)
    
    Q = (e_baseline.T @ W @ e_baseline).item()
    
    B = get_design_matrix(x, e_baseline, observations)
    
    C = B.T @ W @ B
    rhs = B.T @ W @ e_baseline
    
    damping_factor = 1e-3
    C_stabilized = C + damping_factor * np.eye(6)
    
    dx = -np.linalg.solve(C_stabilized, rhs).flatten()
    
    x += dx
    
    pos_step_mag = np.linalg.norm(dx[:3])
    
    if pos_step_mag < tolerance_dx or abs(Q_old - Q) < tolerance_dQ:
        break
        
    Q_old = Q

In [33]:
sigma = np.linalg.inv(C)

![](https://cdn.hackclub.com/019ed6d1-edd3-745d-850b-17f77db9a8e7/image_2026-06-17_235220355.png)

## Tests

In [34]:
print(np.sqrt(np.mean(residual_states**2)) / arcsec_to_rad)
print(np.percentile(np.abs(residual_states) / arcsec_to_rad, [50, 90, 95, 99]))

0.30374581383470606
[0.07820473 0.22138345 0.32252753 1.10775831]


## Evals

In [35]:
mean_residual = np.mean(np.abs(residual_states)) / arcsec_to_rad
print(f"Mean Residual in arcseconds: {mean_residual}")

Mean Residual in arcseconds: 0.13217292974584377


In [50]:
command = "&COMMAND='433'"
obj_data = "&OBJ_DATA='YES'"
make_ephem = "&MAKE_EPHEM='YES'"
ephem_type = "&EPHEM_TYPE='VECTORS'"
center = "&CENTER='@10'" # Solar System Barycenter
ref_plane = "&REF_PLANE='F'" # BODY EQUATOR - Equatorial
out_units = "&OUT_UNITS='KM-S'"
start_time = f"&START_TIME='{sp.et2datetime(t_start).strftime("%Y-%m-%d %H:%M")}'"
stop_time = f"&STOP_TIME='{(sp.et2datetime(t_start) + timedelta(days=1)).strftime("%Y-%m-%d %H:%M")}'"
step_size = "&STEP_SIZE='1d'"

response = requests.get(f"https://ssd.jpl.nasa.gov/api/horizons.api?format=text{command}{obj_data}{make_ephem}{ephem_type}{center}{ref_plane}{out_units}{start_time}{stop_time}{step_size}")